# Paper 2 — GSD Analysis: Final Version
**Topology-Conditioned Structural Reasoning in Hybrid IDS**

This notebook implements the complete, corrected experimental protocol for Paper 2.

**Fixes applied:**
- Fix 1: Composite entity key `Dataset::IP` (IP collision prevention)
- Fix 2: Vectorized random-key reservoir sampling (unbiased, paper-ready)
- Fix 3: `already_prepared` flag (no redundant `prepare_entity_keys`)
- Fix 4: Corrected reference rows — WTMC2021 (FN=5, Recall=0.375, RCI=0.158)
- Fix 5: Temporal-only behavioral channel (FLOW_DURATION_MILLISECONDS)
- Fix 6: Saturation check and P90 exclusion from primary results

**Repository:** KDIS-GSD-Topology-Validity (Paper 2 only)

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

Mounted at /content/drive
✓ Google Drive mounted


## Step 2 — Kaggle API Setup & Download NF-UQ-NIDS-v2

In [ ]:
# Option A: upload kaggle.json manually
# from google.colab import files
# files.upload()

# Option B: load from Drive (recommended after first run)
import os, shutil

KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle.json'

if os.path.exists(KAGGLE_JSON_DRIVE):
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    shutil.copy(KAGGLE_JSON_DRIVE, '/root/.config/kaggle/kaggle.json')
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
    print('✓ kaggle.json loaded from Drive')
else:
    print('kaggle.json not found in Drive — upload manually:')
    from google.colab import files
    files.upload()
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
    # Save to Drive for future sessions
    shutil.copy('/root/.config/kaggle/kaggle.json', KAGGLE_JSON_DRIVE)
    print('✓ kaggle.json saved to Drive for future sessions')

kaggle.json not found in Drive — upload manually:


Saving kaggle.json to kaggle.json
✓ kaggle.json saved to Drive for future sessions


In [ ]:
import subprocess, glob

NF_DIR = '/content/nf_uq_nids'
NF_FILE = f'{NF_DIR}/NF-UQ-NIDS-v2.csv'

if os.path.exists(NF_FILE):
    size_gb = os.path.getsize(NF_FILE) / 1e9
    print(f'✓ File already exists ({size_gb:.1f} GB) — skipping download')
else:
    print('Downloading NF-UQ-NIDS-v2 (~2 GB)...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download',
         '-d', 'aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset',
         '-p', NF_DIR, '--unzip'],
        capture_output=True, text=True
    )
    print(result.stdout[-500:] if result.stdout else '')
    print('✓ Download complete')

Dataset URL: https://www.kaggle.com/datasets/aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset
License(s): CC0-1.0


✓ Download complete


## Step 3 — Configuration & Imports

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import os

# ── Paths ────────────────────────────────────────────────
FILE         = '/content/nf_uq_nids/NF-UQ-NIDS-v2.csv'
OUTPUT_DIR   = '/content/drive/MyDrive/paper2_gsd_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Columns ──────────────────────────────────────────────
SRC_COL      = 'IPV4_SRC_ADDR'
DST_COL      = 'IPV4_DST_ADDR'
LBL_COL      = 'Label'
DATASET_COL  = 'Dataset'
ATTACK_COL   = 'Attack'

# ── Sampling ─────────────────────────────────────────────
MAX_ROWS     = 200_000
CHUNKSIZE    = 250_000
RANDOM_STATE = 42
PERCENTILES  = [90, 95, 97]

# ── Behavioral features (temporal-only — Paper 2 primary) ─
# NOTE: FLOW_DURATION_MILLISECONDS is the only usable temporal
# feature in NF-UQ-NIDS-v2. DURATION_IN/OUT are absent.
# A 7-feature robustness check is available in Appendix section.
BEHAVIOR_FEATURES = ['FLOW_DURATION_MILLISECONDS']

print('✓ Configuration loaded')
print(f'  Output directory: {OUTPUT_DIR}')

✓ Configuration loaded
  Output directory: /content/drive/MyDrive/paper2_gsd_results


## Step 4 — Utility Functions

In [ ]:
def gini_coefficient(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x) & (x >= 0)]
    if len(x) == 0 or x.sum() == 0:
        return 0.0
    x = np.sort(x)
    n = len(x)
    idx = np.arange(1, n + 1)
    return float((2*np.sum(idx*x) - (n+1)*np.sum(x)) / (n*np.sum(x)))

def safe_ratio(a, b):
    if b is None or b == 0 or not np.isfinite(float(b)):
        return np.inf
    return float(a / b)

def make_stratum_key(df, strata_cols):
    return df[strata_cols].astype(str).agg('||'.join, axis=1)

def allocate_quotas(counts, target_n, min_per_stratum=50):
    counts   = counts[counts > 0].copy()
    total    = int(counts.sum())
    target_n = min(int(target_n), total)
    n_strata = len(counts)
    quotas   = pd.Series(0, index=counts.index, dtype=int)
    if n_strata * min_per_stratum > target_n:
        raw = counts / total * target_n
        quotas = np.floor(raw).astype(int)
        rem = target_n - int(quotas.sum())
        if rem > 0:
            for idx in (raw - np.floor(raw)).sort_values(ascending=False).index[:rem]:
                quotas.loc[idx] += 1
        return quotas.clip(lower=0, upper=counts).astype(int)
    for idx, c in counts.items():
        quotas.loc[idx] = min(min_per_stratum, int(c))
    remaining = target_n - int(quotas.sum())
    if remaining > 0:
        cap = (counts - quotas).clip(lower=0)
        cap_total = int(cap.sum())
        if cap_total > 0:
            raw_extra = cap / cap_total * remaining
            extra = np.floor(raw_extra).astype(int)
            rem2 = remaining - int(extra.sum())
            if rem2 > 0:
                for idx in (raw_extra - np.floor(raw_extra)).sort_values(ascending=False).index[:rem2]:
                    extra.loc[idx] += 1
            quotas = (quotas + extra).clip(lower=0, upper=counts)
    return quotas.astype(int)

def classification_metrics(pred, truth, base_rate):
    pred, truth = pred.astype(bool), truth.astype(bool)
    tp   = int((pred & truth).sum())
    fp   = int((pred & ~truth).sum())
    fn   = int((~pred & truth).sum())
    tn   = int((~pred & ~truth).sum())
    prec = tp/(tp+fp) if (tp+fp) else 0.0
    rec  = tp/(tp+fn) if (tp+fn) else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    fpr  = fp/(fp+tn) if (fp+tn) else 0.0
    lift = (tp/int(pred.sum())/base_rate) if int(pred.sum()) and base_rate else 0.0
    return {'Flagged':int(pred.sum()),'TP':tp,'FP':fp,'FN':fn,'TN':tn,
            'Precision':prec,'Recall':rec,'F1':f1,'FPR':fpr,'Lift':lift}

def jaccard_rci(S, B):
    S, B   = S.astype(bool), B.astype(bool)
    inter  = int((S & B).sum())
    union  = int((S | B).sum())
    rci    = inter/union if union else 0.0
    sr, br = float(S.mean()), float(B.mean())
    denom  = sr + br - sr*br
    rnd    = (sr*br)/denom if denom > 0 else 0.0
    return rci, rnd, rci/rnd if rnd > 0 else np.inf

print('✓ Utility functions defined')

✓ Utility functions defined


## Step 5 — Stratified Sampling
**Method:** Vectorized random-key reservoir sampling (Fix 2)
**Stratification:** Dataset × Attack type
**Guarantee:** Every row has equal probability k/N of selection

In [ ]:
peek = pd.read_csv(FILE, nrows=5, low_memory=False)
strata_cols = [DATASET_COL, ATTACK_COL]
usecols = list(dict.fromkeys(
    [SRC_COL, DST_COL, LBL_COL, DATASET_COL, ATTACK_COL]
    + BEHAVIOR_FEATURES
))

rng = np.random.default_rng(RANDOM_STATE)

# ── Pass 1: count strata ──────────────────────────────
print('Pass 1/2: counting strata...')
counts = defaultdict(int)
for chunk in pd.read_csv(FILE,
        usecols=[c for c in strata_cols if c in peek.columns],
        chunksize=CHUNKSIZE, low_memory=False):
    for k, v in make_stratum_key(chunk, strata_cols).value_counts().items():
        counts[k] += int(v)

counts  = pd.Series(counts).sort_values(ascending=False)
quotas  = allocate_quotas(counts, target_n=MAX_ROWS, min_per_stratum=50)
print(f'Total rows in file: {int(counts.sum()):,}')
print(f'Sample quota:       {int(quotas.sum()):,}')
print('\nTop 10 strata:')
print(pd.DataFrame({'count':counts,'quota':quotas}).head(10).to_string())

# ── Pass 2: vectorized random-key reservoir ───────────
print('\nPass 2/2: random-key reservoir sampling...')
reservoirs = {}
for chunk in pd.read_csv(FILE, usecols=usecols,
                          chunksize=CHUNKSIZE, low_memory=False):
    chunk = chunk.dropna(subset=[SRC_COL, DST_COL, LBL_COL]).copy()
    chunk['_stratum'] = make_stratum_key(chunk, strata_cols)
    chunk['_rand']    = rng.random(len(chunk))
    for stratum, grp in chunk.groupby('_stratum', sort=False):
        quota = int(quotas.get(stratum, 0))
        if quota <= 0:
            continue
        grp = grp.copy()
        if len(grp) > quota:
            grp = grp.nsmallest(quota, '_rand')
        combined = (pd.concat([reservoirs[stratum], grp], ignore_index=True)
                    if stratum in reservoirs else grp)
        if len(combined) > quota:
            combined = combined.nsmallest(quota, '_rand')
        reservoirs[stratum] = combined

df = (pd.concat(reservoirs.values(), ignore_index=True)
        .drop(columns=['_stratum', '_rand'], errors='ignore')
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True))

print(f'\n✓ Sample size: {len(df):,}')
print('\nDataset distribution:')
print(df[DATASET_COL].value_counts().to_string())

Pass 1/2: counting strata...
Total rows in file: 75,987,976
Sample quota:       200,000

Top 10 strata:
                                  count  quota
NF-BoT-IoT-v2||DDoS            18331847  47914
NF-BoT-IoT-v2||DoS             16673183  43583
NF-CSE-CIC-IDS2018-v2||Benign  16635567  43485
NF-ToN-IoT-v2||Benign           6099469  15976
NF-ToN-IoT-v2||scanning         3781419   9923
NF-BoT-IoT-v2||Reconnaissance   2620999   6893
NF-ToN-IoT-v2||xss              2455020   6460
NF-UNSW-NB15-v2||Benign         2295222   6043
NF-ToN-IoT-v2||DDoS             2026234   5340
NF-CSE-CIC-IDS2018-v2||DDoS     1390270   3680

Pass 2/2: random-key reservoir sampling...

✓ Sample size: 200,000

Dataset distribution:
Dataset
NF-BoT-IoT-v2            98848
NF-CSE-CIC-IDS2018-v2    49682
NF-ToN-IoT-v2            44730
NF-UNSW-NB15-v2           6740


## Step 6 — Composite Entity Keys (Fix 1: IP Collision Prevention)
Entity key = `Dataset::IP` to prevent merging identical private IPs across sub-datasets.

In [ ]:
# Fix 1: Composite key Dataset::IP
df[DATASET_COL] = df[DATASET_COL].astype(str).str.strip()
df[SRC_COL]     = df[SRC_COL].astype(str).str.strip()
df[DST_COL]     = df[DST_COL].astype(str).str.strip()
df['src_key']   = df[DATASET_COL] + '::' + df[SRC_COL]
df['dst_key']   = df[DATASET_COL] + '::' + df[DST_COL]
df['is_attack'] = (pd.to_numeric(df[LBL_COL], errors='coerce')
                     .fillna(0).astype(int).eq(1))

print('✓ Composite entity keys created')
print(f'  Sample src_key: {df["src_key"].iloc[0]}')
print(f'  Attack rate:    {df["is_attack"].mean():.4f}')

✓ Composite entity keys created
  Sample src_key: NF-BoT-IoT-v2::192.168.100.147
  Attack rate:    0.6705


## Step 7 — GSD Analysis: Combined + Per Sub-dataset

In [ ]:
def compute_entity_table(df_in):
    edges   = df_in[['src_key','dst_key']].dropna().drop_duplicates()
    out_deg = edges.groupby('src_key')['dst_key'].nunique().rename('out_degree')
    src_att = (df_in.groupby('src_key')['is_attack']
                    .max().astype(int).rename('src_is_attacker'))
    entity  = pd.DataFrame({'out_degree': out_deg}).join(src_att, how='left')
    entity['src_is_attacker'] = entity['src_is_attacker'].fillna(0).astype(int)
    return entity

def compute_gsd(entity, dataset_name):
    od_nz   = entity.loc[entity['out_degree'] > 0, 'out_degree'].astype(float)
    total_e = len(entity)
    total_a = int(entity['src_is_attacker'].sum())
    base    = total_a / total_e if total_e else 0.0
    if len(od_nz) == 0:
        return {'Dataset': dataset_name, 'GSD_Level': 'Undefined'}
    med  = float(np.median(od_nz))
    p90  = float(np.percentile(od_nz, 90))
    p95  = float(np.percentile(od_nz, 95))
    p97  = float(np.percentile(od_nz, 97))
    mx   = float(od_nz.max())
    cv   = safe_ratio(od_nz.std(), od_nz.mean())
    gini = gini_coefficient(od_nz.values)
    p95m = safe_ratio(p95, med)
    tc   = bool(np.isclose(p90, p95) and np.isclose(p95, p97))
    # NOTE: boundaries are empirically derived, not universal
    if   (not tc) and gini >= 0.75 and p95m >= 100: level = 'High'
    elif (not tc) and gini >= 0.50 and p95m >= 10:  level = 'Medium'
    else:                                             level = 'Low'
    return {'Dataset': dataset_name, 'Entities': total_e,
            'Attacker_Entities': total_a, 'Base_Rate': round(base,6),
            'Nonzero_Src': len(od_nz), 'Median': round(med,2),
            'P95': round(p95,2), 'Max': round(mx,2),
            'CV': round(cv,4), 'Gini': round(gini,4),
            'P95_Median': round(p95m,2),
            'Threshold_Collapse': tc, 'GSD_Level': level}

# Run GSD on combined + per sub-dataset
gsd_results = []
for ds_name, sub in [('NF-UQ-NIDS-v2_COMBINED', df)] + [
        (str(n), s) for n, s in df.groupby(DATASET_COL) if len(s) >= 100]:
    entity_tmp = compute_entity_table(sub)
    gsd_results.append(compute_gsd(entity_tmp, ds_name))

# Add reference rows (Fix 4: corrected WTMC2021 values)
gsd_results += [
    {'Dataset':'WTMC2021 [Ref-High]','Entities':19060,'Attacker_Entities':8,
     'Base_Rate':0.000420,'Nonzero_Src':47,'Median':4.0,'P95':188577.8,
     'Max':442056.0,'CV':1.9736,'Gini':0.7994,'P95_Median':47144.45,
     'Threshold_Collapse':False,'GSD_Level':'High'},
    {'Dataset':'UNSW-NB15 [Ref-Low]','Entities':47,'Attacker_Entities':4,
     'Base_Rate':0.085106,'Nonzero_Src':42,'Median':10.0,'P95':10.0,
     'Max':11.0,'CV':0.4695,'Gini':0.2555,'P95_Median':1.18,
     'Threshold_Collapse':True,'GSD_Level':'Low'},
]

gsd_df = pd.DataFrame(gsd_results)
print('=== GSD SUMMARY ===')
print(gsd_df[['Dataset','Entities','Attacker_Entities','Base_Rate',
              'Median','P95','Gini','P95_Median',
              'Threshold_Collapse','GSD_Level']].to_string(index=False))

gsd_df.to_csv(f'{OUTPUT_DIR}/gsd_summary_final.csv',
              index=False, encoding='utf-8-sig')
print(f'\n✓ Saved: gsd_summary_final.csv')

=== GSD SUMMARY ===
               Dataset  Entities  Attacker_Entities  Base_Rate  Median      P95   Gini  P95_Median  Threshold_Collapse GSD_Level
NF-UQ-NIDS-v2_COMBINED      5389                 73   0.013546     1.0     20.6 0.6772       20.60               False    Medium
         NF-BoT-IoT-v2        10                  8   0.800000     5.0      7.1 0.2208        1.42               False       Low
 NF-CSE-CIC-IDS2018-v2      5245                 40   0.007626     1.0     21.0 0.6675       21.00               False    Medium
         NF-ToN-IoT-v2       108                 21   0.194444     1.0     17.0 0.8445       17.00               False    Medium
       NF-UNSW-NB15-v2        26                  4   0.153846    10.0     10.0 0.3588        1.00                True       Low
   WTMC2021 [Ref-High]     19060                  8   0.000420     4.0 188577.8 0.7994    47144.45               False      High
   UNSW-NB15 [Ref-Low]        47                  4   0.085106    10.0     10

## Step 8 — KDIS Pipeline: Temporal-Only Behavioral Channel
**Primary analysis** uses FLOW_DURATION_MILLISECONDS only for methodological
consistency with Paper 1. See Step 10 for 7-feature robustness check.

**Primary results:** P95 and P97 only (P90 excluded due to behavioral saturation).

In [ ]:
def run_kdis(df_in, dataset_name, features):
    for f in features:
        df_in[f] = pd.to_numeric(df_in[f], errors='coerce')
    df_in = df_in.dropna(subset=features).copy()
    entity    = compute_entity_table(df_in)
    truth     = entity['src_is_attacker'].astype(bool)
    base_rate = float(truth.mean()) if len(truth) else 0.0
    od_nz     = entity.loc[entity['out_degree']>0,'out_degree'].astype(float)
    if len(od_nz) == 0:
        return pd.DataFrame()
    benign_df = df_in[~df_in['is_attack']]
    if benign_df.empty:
        benign_df = df_in.copy()
    rows = []
    for p in PERCENTILES:
        thr    = {f: float(benign_df[f].quantile(p/100)) for f in features}
        b_flow = np.zeros(len(df_in), dtype=bool)
        for f, t in thr.items():
            b_flow |= df_in[f].to_numpy() >= t
        b_rate = float(b_flow.mean())
        B_src  = (pd.DataFrame({'src_key':df_in['src_key'].to_numpy(),
                                'B':b_flow.astype(int)})
                    .groupby('src_key')['B'].max()
                    .reindex(entity.index).fillna(0).astype(bool))
        b_entity_rate = float(B_src.mean())
        s_thr  = float(od_nz.quantile(p/100))
        S_out  = entity['out_degree'].astype(float) >= s_thr
        Hybrid = B_src & S_out
        rci, rnd, rci_lift = jaccard_rci(S_out, B_src)
        for name, pred in [('B-only',B_src),('S-only',S_out),('Hybrid',Hybrid)]:
            m = classification_metrics(
                pred.reset_index(drop=True),
                truth.reset_index(drop=True), base_rate)
            rows.append({
                'Dataset':dataset_name,'Threshold':f'P{p}','Config':name,
                'B_entity_rate':round(b_entity_rate,4),
                'B_saturated':b_entity_rate>=0.99,
                'S_threshold':s_thr,
                'RCI':rci if name=='Hybrid' else np.nan,
                'RCI_Lift':rci_lift if name=='Hybrid' else np.nan,
                **m})
    return pd.DataFrame(rows)

# Run on combined + per sub-dataset
all_kdis = []
for ds_name, sub in [('NF-UQ-NIDS-v2_COMBINED', df)] + [
        (str(n), s) for n, s in df.groupby(DATASET_COL) if len(s) >= 100]:
    print(f'Analyzing: {ds_name} ({len(sub):,} rows)')
    res = run_kdis(sub.copy(), ds_name, BEHAVIOR_FEATURES)
    if not res.empty:
        all_kdis.append(res)

kdis_df   = pd.concat(all_kdis, ignore_index=True) if all_kdis else pd.DataFrame()
hybrid_df = kdis_df[kdis_df['Config']=='Hybrid'].copy()

# Fix 4: corrected reference rows (WTMC2021: FN=5, Recall=0.375, RCI=0.158)
ref_hybrid = pd.DataFrame([
    {'Dataset':'WTMC2021 [Ref-High]','Threshold':'P95','Flagged':3,
     'TP':3,'FP':0,'FN':5,'Precision':1.000,'Recall':0.375,'F1':0.545,
     'FPR':0.000,'Lift':2383.0,'RCI':0.158,'RCI_Lift':1162.0,
     'B_entity_rate':None,'B_saturated':False},
    {'Dataset':'UNSW-NB15 [Ref-Low]','Threshold':'P95','Flagged':19,
     'TP':4,'FP':15,'FN':0,'Precision':0.211,'Recall':1.000,'F1':0.348,
     'FPR':0.349,'Lift':2.47,'RCI':0.500,'RCI_Lift':1.36,
     'B_entity_rate':None,'B_saturated':False},
])

hybrid_final = pd.concat([hybrid_df, ref_hybrid], ignore_index=True)

# Fix 6: Primary results — P95/P97, non-saturated only
primary = hybrid_df[
    hybrid_df['Threshold'].isin(['P95','P97']) &
    ~hybrid_df['B_saturated']
].copy()

print('\n=== SATURATION CHECK ===')
print(kdis_df[kdis_df['Config']=='B-only']
      [['Dataset','Threshold','Flagged','B_entity_rate','B_saturated']]
      .to_string(index=False))

print('\n=== PRIMARY RESULTS (P95/P97, non-saturated) ===')
print(primary[['Dataset','Threshold','Flagged','TP','FP','FN',
               'Precision','Recall','FPR','Lift','RCI','RCI_Lift']]
      .round(4).to_string(index=False))

print('\n=== HYBRID TABLE WITH REFERENCES ===')
print(hybrid_final[['Dataset','Threshold','Flagged','TP','FP','FN',
                    'Precision','Recall','FPR','Lift','RCI','RCI_Lift']]
      .round(4).to_string(index=False))

kdis_df.to_csv(f'{OUTPUT_DIR}/kdis_full_results.csv',
               index=False, encoding='utf-8-sig')
hybrid_final.to_csv(f'{OUTPUT_DIR}/hybrid_results_final.csv',
                    index=False, encoding='utf-8-sig')
primary.to_csv(f'{OUTPUT_DIR}/primary_results_p95_p97.csv',
               index=False, encoding='utf-8-sig')
print('\n✓ Results saved to Drive')

Analyzing: NF-UQ-NIDS-v2_COMBINED (200,000 rows)
Analyzing: NF-BoT-IoT-v2 (98,848 rows)
Analyzing: NF-CSE-CIC-IDS2018-v2 (49,682 rows)
Analyzing: NF-ToN-IoT-v2 (44,730 rows)
Analyzing: NF-UNSW-NB15-v2 (6,740 rows)

=== SATURATION CHECK ===
               Dataset Threshold  Flagged  B_entity_rate  B_saturated
NF-UQ-NIDS-v2_COMBINED       P90     5389         1.0000         True
NF-UQ-NIDS-v2_COMBINED       P95      591         0.1097        False
NF-UQ-NIDS-v2_COMBINED       P97      541         0.1004        False
         NF-BoT-IoT-v2       P90        7         0.7000        False
         NF-BoT-IoT-v2       P95        7         0.7000        False
         NF-BoT-IoT-v2       P97        7         0.7000        False
 NF-CSE-CIC-IDS2018-v2       P90     5245         1.0000         True
 NF-CSE-CIC-IDS2018-v2       P95     5245         1.0000         True
 NF-CSE-CIC-IDS2018-v2       P97      477         0.0909        False
         NF-ToN-IoT-v2       P90      108         1.0000    

/tmp/ipykernel_1208/837303502.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hybrid_final = pd.concat([hybrid_df, ref_hybrid], ignore_index=True)



✓ Results saved to Drive


## Step 9 — Cross-Paradigm Invariance Test
Tests whether S ⊂ B ∩ IF holds in NF-ToN-IoT-v2 (Medium GSD).
Compares with WTMC2021 finding S ⊂ B ∩ RF ∩ IF (Paper 1, High GSD).

**IF:** Unsupervised, no training needed.
**RF:** Trained on NF-BoT-IoT-v2, applied cross-sub-dataset to NF-ToN-IoT-v2.

In [ ]:
from sklearn.ensemble import IsolationForest, RandomForestClassifier

PERCENTILE = 95
FEATS_7 = ['FLOW_DURATION_MILLISECONDS','IN_BYTES','OUT_BYTES',
           'IN_PKTS','OUT_PKTS','SRC_TO_DST_SECOND_BYTES','DST_TO_SRC_SECOND_BYTES']

sub_ton = df[df[DATASET_COL]=='NF-ToN-IoT-v2'].copy()
sub_bot = df[df[DATASET_COL]=='NF-BoT-IoT-v2'].copy()

for f in FEATS_7:
    sub_ton[f] = pd.to_numeric(sub_ton[f], errors='coerce')
    sub_bot[f] = pd.to_numeric(sub_bot[f], errors='coerce')

sub_ton = sub_ton.dropna(subset=FEATS_7).copy()
sub_bot = sub_bot.dropna(subset=FEATS_7).copy()

# Entity space
entity_ton = compute_entity_table(sub_ton)
truth_ton  = entity_ton['src_is_attacker'].astype(bool)
base_ton   = float(truth_ton.mean())
od_nz_ton  = entity_ton.loc[entity_ton['out_degree']>0,'out_degree'].astype(float)
s_thr_ton  = float(od_nz_ton.quantile(PERCENTILE/100))
S_out_ton  = entity_ton['out_degree'].astype(float) >= s_thr_ton

# B channel
benign_ton = sub_ton[~sub_ton['is_attack']]
thr_b = {f: float(benign_ton[f].quantile(PERCENTILE/100)) for f in FEATS_7}
b_flow = np.zeros(len(sub_ton), dtype=bool)
for f, t in thr_b.items():
    b_flow |= sub_ton[f].to_numpy() >= t
B_src_ton = (pd.DataFrame({'src_key':sub_ton['src_key'].to_numpy(),'B':b_flow.astype(int)})
               .groupby('src_key')['B'].max()
               .reindex(entity_ton.index).fillna(0).astype(bool))

# Isolation Forest
iso = IsolationForest(n_estimators=100, random_state=42, contamination='auto')
iso.fit(sub_ton[FEATS_7].fillna(0))
sub_ton['IF_flag'] = (iso.predict(sub_ton[FEATS_7].fillna(0))==-1).astype(int)
IF_src_ton = (sub_ton.groupby('src_key')['IF_flag'].max()
                .reindex(entity_ton.index).fillna(0).astype(bool))

# Random Forest (train on NF-BoT-IoT-v2)
RF_src_ton = None
if len(sub_bot) >= 100:
    rf = RandomForestClassifier(n_estimators=50, random_state=42,
                                 class_weight='balanced')
    rf.fit(sub_bot[FEATS_7].fillna(0), sub_bot['is_attack'].astype(int))
    sub_ton['RF_flag'] = rf.predict(sub_ton[FEATS_7].fillna(0))
    RF_src_ton = (sub_ton.groupby('src_key')['RF_flag'].max()
                    .reindex(entity_ton.index).fillna(0).astype(bool))

# Sets
S  = set(entity_ton.index[S_out_ton])
B  = set(B_src_ton[B_src_ton].index)
IF = set(IF_src_ton[IF_src_ton].index)
RF = set(RF_src_ton[RF_src_ton].index) if RF_src_ton is not None else set()
ATK= set(entity_ton.index[truth_ton])

print('=== CROSS-PARADIGM INVARIANCE — NF-ToN-IoT-v2 (Medium GSD) ===')
print(f'S entities:      {S}')
print(f'S ∩ ATK:         {S & ATK}  (all attackers: {ATK})')
print(f'S ⊂ B:           {S.issubset(B)}')
print(f'S ⊂ IF:          {S.issubset(IF)}')
print(f'S ⊂ B ∩ IF:      {S.issubset(B & IF)}')
if RF:
    print(f'S ⊂ RF:          {S.issubset(RF)}')
    print(f'S ⊂ B ∩ RF ∩ IF: {S.issubset(B & RF & IF)}')

print('\n=== COMPARISON WITH WTMC2021 (Paper 1 — High GSD) ===')
print(f'{'Environment':<25} {'GSD':>8} {'S⊂B':>6} {'S⊂IF':>6} {'S⊂B∩RF∩IF':>12}')
print('-'*60)
print(f'{'WTMC2021 [Ref-High]':<25} {'High':>8} {'Yes':>6} {'Yes':>6} {'Yes':>12}')
print(f'{'NF-ToN-IoT-v2':<25} {'Med/H':>8} '
      f"{'Yes' if S.issubset(B) else 'No':>6} "
      f"{'Yes' if S.issubset(IF) else 'No':>6} "
      f"{'Yes' if (RF and S.issubset(B & RF & IF)) else 'N/A':>12}")

import json
with open(f'{OUTPUT_DIR}/cross_paradigm_ton_iot.json','w') as f_out:
    json.dump({'S':list(S),'ATK':list(ATK),
               'S_subset_B':S.issubset(B),
               'S_subset_IF':S.issubset(IF),
               'S_subset_B_IF':S.issubset(B & IF),
               'S_subset_B_RF_IF':(S.issubset(B & RF & IF) if RF else 'N/A')},
              f_out, indent=2, default=str)
print('\n✓ Cross-paradigm results saved')

KeyError: 'IN_BYTES'

In [ ]:
# ============================================================
# Fix: reload NF-ToN-IoT-v2 and NF-BoT-IoT-v2 with all 7 features
# ============================================================

import pandas as pd
import numpy as np

FILE    = '/content/nf_uq_nids/NF-UQ-NIDS-v2.csv'
FEATS_7 = ['FLOW_DURATION_MILLISECONDS','IN_BYTES','OUT_BYTES',
           'IN_PKTS','OUT_PKTS','SRC_TO_DST_SECOND_BYTES','DST_TO_SRC_SECOND_BYTES']

LOAD_COLS = [SRC_COL, DST_COL, LBL_COL, DATASET_COL] + FEATS_7

print("Loading NF-ToN-IoT-v2 and NF-BoT-IoT-v2 from file...")

# قراءة الملف بـ chunks وتصفية sub-datasets المطلوبة فقط
chunks_ton, chunks_bot = [], []

for chunk in pd.read_csv(FILE, usecols=LOAD_COLS,
                          chunksize=250_000, low_memory=False):
    mask_ton = chunk[DATASET_COL] == 'NF-ToN-IoT-v2'
    mask_bot = chunk[DATASET_COL] == 'NF-BoT-IoT-v2'
    if mask_ton.any():
        chunks_ton.append(chunk[mask_ton])
    if mask_bot.any():
        chunks_bot.append(chunk[mask_bot])

sub_ton_full = pd.concat(chunks_ton, ignore_index=True)
sub_bot_full = pd.concat(chunks_bot, ignore_index=True)

# Stratified sample 50K من كل منهما
sub_ton = (sub_ton_full.groupby('Attack', group_keys=False)
                       .apply(lambda x: x.sample(
                           min(len(x), max(50, int(50_000 * len(x)/len(sub_ton_full)))),
                           random_state=42))
                       .reset_index(drop=True))

sub_bot = sub_bot_full.sample(
    min(50_000, len(sub_bot_full)), random_state=42
).reset_index(drop=True)

# إضافة المفاتيح
for sub in [sub_ton, sub_bot]:
    sub[SRC_COL]     = sub[SRC_COL].astype(str).str.strip()
    sub[DST_COL]     = sub[DST_COL].astype(str).str.strip()
    sub[DATASET_COL] = sub[DATASET_COL].astype(str).str.strip()
    sub['src_key']   = sub[DATASET_COL] + '::' + sub[SRC_COL]
    sub['dst_key']   = sub[DATASET_COL] + '::' + sub[DST_COL]
    sub['is_attack'] = (pd.to_numeric(sub[LBL_COL], errors='coerce')
                          .fillna(0).astype(int).eq(1))

for f in FEATS_7:
    sub_ton[f] = pd.to_numeric(sub_ton[f], errors='coerce')
    sub_bot[f] = pd.to_numeric(sub_bot[f], errors='coerce')

sub_ton = sub_ton.dropna(subset=FEATS_7).copy()
sub_bot = sub_bot.dropna(subset=FEATS_7).copy()

print(f"✓ NF-ToN-IoT-v2:  {len(sub_ton):,} rows, "
      f"{sub_ton['src_key'].nunique()} unique entities")
print(f"✓ NF-BoT-IoT-v2:  {len(sub_bot):,} rows, "
      f"{sub_bot['src_key'].nunique()} unique entities")
print(f"  Columns: {[c for c in sub_ton.columns if c in FEATS_7]}")

Loading NF-ToN-IoT-v2 and NF-BoT-IoT-v2 from file...


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Memory-efficient fix: filter by known entities only
# ============================================================

import pandas as pd
import numpy as np

FILE    = '/content/nf_uq_nids/NF-UQ-NIDS-v2.csv'
FEATS_7 = ['FLOW_DURATION_MILLISECONDS','IN_BYTES','OUT_BYTES',
           'IN_PKTS','OUT_PKTS','SRC_TO_DST_SECOND_BYTES','DST_TO_SRC_SECOND_BYTES']
LOAD_COLS = [SRC_COL, DST_COL, LBL_COL, DATASET_COL] + FEATS_7

# الكيانات المعروفة من df المحمّل مسبقاً
ton_ips = set(df.loc[df[DATASET_COL]=='NF-ToN-IoT-v2', SRC_COL].unique())
bot_ips = set(df.loc[df[DATASET_COL]=='NF-BoT-IoT-v2', SRC_COL].unique())
target_ips = ton_ips | bot_ips

print(f"NF-ToN-IoT-v2 known IPs: {len(ton_ips)}")
print(f"NF-BoT-IoT-v2 known IPs: {len(bot_ips)}")
print("Reading file (filtered rows only)...")

chunks_ton, chunks_bot = [], []
MAX_TON, MAX_BOT = 30_000, 30_000

for chunk in pd.read_csv(FILE, usecols=LOAD_COLS,
                          chunksize=500_000, low_memory=False):

    # فلترة مزدوجة: dataset + IP معروف
    mask_ton = ((chunk[DATASET_COL] == 'NF-ToN-IoT-v2') &
                (chunk[SRC_COL].isin(ton_ips)))
    mask_bot = ((chunk[DATASET_COL] == 'NF-BoT-IoT-v2') &
                (chunk[SRC_COL].isin(bot_ips)))

    if mask_ton.any() and sum(len(c) for c in chunks_ton) < MAX_TON:
        chunks_ton.append(chunk[mask_ton])
    if mask_bot.any() and sum(len(c) for c in chunks_bot) < MAX_BOT:
        chunks_bot.append(chunk[mask_bot])

    # توقف مبكر حين نجمع ما يكفي
    if (sum(len(c) for c in chunks_ton) >= MAX_TON and
        sum(len(c) for c in chunks_bot) >= MAX_BOT):
        break

sub_ton = pd.concat(chunks_ton, ignore_index=True).head(MAX_TON)
sub_bot = pd.concat(chunks_bot, ignore_index=True).head(MAX_BOT)

# إضافة المفاتيح وتنظيف
for sub in [sub_ton, sub_bot]:
    sub[SRC_COL]     = sub[SRC_COL].astype(str).str.strip()
    sub[DST_COL]     = sub[DST_COL].astype(str).str.strip()
    sub[DATASET_COL] = sub[DATASET_COL].astype(str).str.strip()
    sub['src_key']   = sub[DATASET_COL] + '::' + sub[SRC_COL]
    sub['dst_key']   = sub[DATASET_COL] + '::' + sub[DST_COL]
    sub['is_attack'] = (pd.to_numeric(sub[LBL_COL], errors='coerce')
                          .fillna(0).astype(int).eq(1))
    for f in FEATS_7:
        sub[f] = pd.to_numeric(sub[f], errors='coerce')

sub_ton = sub_ton.dropna(subset=FEATS_7).copy()
sub_bot = sub_bot.dropna(subset=FEATS_7).copy()

print(f"✓ NF-ToN-IoT-v2: {len(sub_ton):,} rows | "
      f"{sub_ton['src_key'].nunique()} entities")
print(f"✓ NF-BoT-IoT-v2: {len(sub_bot):,} rows | "
      f"{sub_bot['src_key'].nunique()} entities")

NF-ToN-IoT-v2 known IPs: 108
NF-BoT-IoT-v2 known IPs: 10
Reading file (filtered rows only)...
✓ NF-ToN-IoT-v2: 30,000 rows | 37 entities
✓ NF-BoT-IoT-v2: 30,000 rows | 10 entities


In [ ]:
# ============================================================
# Fix: handle overflow in RF + get results
# ============================================================
import numpy as np

def clean_features(X):
    """Replace inf and clip extreme values for RF compatibility."""
    X = X.copy().astype(float)
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X.fillna(0, inplace=True)
    # Clip to float32 safe range
    X = X.clip(-3.4e38, 3.4e38)
    return X

# Random Forest مع بيانات نظيفة
if len(sb) >= 100:
    X_train = clean_features(sb[FEATS_7])
    y_train = sb['is_attack'].astype(int)
    X_test  = clean_features(st[FEATS_7])

    rf = RandomForestClassifier(n_estimators=50, random_state=42,
                                 class_weight='balanced')
    rf.fit(X_train, y_train)
    st['RF_flag'] = rf.predict(X_test)
    RF  = (st.groupby('src_key')['RF_flag'].max()
             .reindex(ent.index).fillna(0).astype(bool))
    RFs = set(RF[RF].index)
    print(f"✓ RF trained and applied | RF flagged: {len(RFs)} entities")
else:
    RFs = set()
    print("RF skipped — insufficient BoT data")

# النتائج النهائية
print('\n=== CROSS-PARADIGM INVARIANCE — NF-ToN-IoT-v2 ===')
print(f'S  = {Ss}')
print(f'S ∩ ATK    = {Ss & ATK}  |  ATK = {ATK}')
print(f'S ⊂ B      = {Ss.issubset(Bs)}')
print(f'S ⊂ IF     = {Ss.issubset(IFs)}')
print(f'S ⊂ B∩IF   = {Ss.issubset(Bs & IFs)}')
if RFs:
    print(f'S ⊂ RF     = {Ss.issubset(RFs)}')
    print(f'S ⊂ B∩RF∩IF= {Ss.issubset(Bs & RFs & IFs)}')

print('\n=== COMPARISON TABLE ===')
print(f"{'Environment':<25} {'GSD':>8} {'S⊂B':>6} {'S⊂IF':>6} {'S⊂B∩RF∩IF':>12}")
print('-'*60)
print(f"{'WTMC2021 [Ref-High]':<25} {'High':>8} {'Yes':>6} {'Yes':>6} {'Yes':>12}")
print(f"{'NF-ToN-IoT-v2':<25} {'Med/H':>8} "
      f"{'Yes' if Ss.issubset(Bs) else 'No':>6} "
      f"{'Yes' if Ss.issubset(IFs) else 'No':>6} "
      f"{'Yes' if (RFs and Ss.issubset(Bs&RFs&IFs)) else 'N/A':>12}")

# حفظ
import json, os
OUTPUT_DIR = '/content/drive/MyDrive/paper2_gsd_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(f'{OUTPUT_DIR}/cross_paradigm_ton_iot.json','w') as fout:
    json.dump({'S':list(Ss),'ATK':list(ATK),
               'S_subset_B':   Ss.issubset(Bs),
               'S_subset_IF':  Ss.issubset(IFs),
               'S_subset_B_IF':Ss.issubset(Bs & IFs),
               'S_subset_B_RF_IF': Ss.issubset(Bs&RFs&IFs) if RFs else 'N/A'},
              fout, indent=2, default=str)
print('\n✓ Saved')

✓ RF trained and applied | RF flagged: 35 entities

=== CROSS-PARADIGM INVARIANCE — NF-ToN-IoT-v2 ===


NameError: name 'Ss' is not defined

In [ ]:
# ============================================================
# Cross-Paradigm Invariance — Complete Self-Contained
# ============================================================

from sklearn.ensemble import IsolationForest, RandomForestClassifier
import pandas as pd, numpy as np, json, os

FILE    = '/content/nf_uq_nids/NF-UQ-NIDS-v2.csv'
FEATS_7 = ['FLOW_DURATION_MILLISECONDS','IN_BYTES','OUT_BYTES',
           'IN_PKTS','OUT_PKTS','SRC_TO_DST_SECOND_BYTES','DST_TO_SRC_SECOND_BYTES']
COLS    = ['IPV4_SRC_ADDR','IPV4_DST_ADDR','Label','Dataset'] + FEATS_7
P       = 95
OUTPUT_DIR = '/content/drive/MyDrive/paper2_gsd_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def clean(X):
    X = X.copy().astype(float)
    X.replace([np.inf,-np.inf], np.nan, inplace=True)
    X.fillna(0, inplace=True)
    return X.clip(-3.4e38, 3.4e38)

# ── 1) Load ───────────────────────────────────────────────
ton_ips = set(df.loc[df['Dataset']=='NF-ToN-IoT-v2','IPV4_SRC_ADDR'].astype(str).str.strip())
bot_ips = set(df.loc[df['Dataset']=='NF-BoT-IoT-v2','IPV4_SRC_ADDR'].astype(str).str.strip())
print(f"Loading {len(ton_ips)} ToN IPs + {len(bot_ips)} BoT IPs...")

ct, cb = [], []
for chunk in pd.read_csv(FILE, usecols=COLS, chunksize=500_000, low_memory=False):
    src = chunk['IPV4_SRC_ADDR'].astype(str).str.strip()
    ds  = chunk['Dataset'].astype(str)
    if (ds=='NF-ToN-IoT-v2').any() and sum(len(c) for c in ct) < 30_000:
        ct.append(chunk[(ds=='NF-ToN-IoT-v2') & src.isin(ton_ips)])
    if (ds=='NF-BoT-IoT-v2').any() and sum(len(c) for c in cb) < 30_000:
        cb.append(chunk[(ds=='NF-BoT-IoT-v2') & src.isin(bot_ips)])
    if sum(len(c) for c in ct)>=30_000 and sum(len(c) for c in cb)>=30_000:
        break

st = pd.concat(ct, ignore_index=True).head(30_000)
sb = pd.concat(cb, ignore_index=True).head(30_000)

for s in [st, sb]:
    s['IPV4_SRC_ADDR'] = s['IPV4_SRC_ADDR'].astype(str).str.strip()
    s['IPV4_DST_ADDR'] = s['IPV4_DST_ADDR'].astype(str).str.strip()
    s['src_key'] = s['Dataset'].astype(str) + '::' + s['IPV4_SRC_ADDR']
    s['dst_key'] = s['Dataset'].astype(str) + '::' + s['IPV4_DST_ADDR']
    s['is_attack'] = pd.to_numeric(s['Label'],errors='coerce').fillna(0).astype(int).eq(1)
    for f in FEATS_7:
        s[f] = pd.to_numeric(s[f], errors='coerce')

st = st.dropna(subset=FEATS_7).copy()
sb = sb.dropna(subset=FEATS_7).copy()
print(f"✓ ToN: {len(st):,} rows | {st['src_key'].nunique()} entities")
print(f"✓ BoT: {len(sb):,} rows | {sb['src_key'].nunique()} entities")

# ── 2) Entity space ───────────────────────────────────────
edges   = st[['src_key','dst_key']].dropna().drop_duplicates()
out_deg = edges.groupby('src_key')['dst_key'].nunique().rename('out_degree')
src_att = st.groupby('src_key')['is_attack'].max().astype(int).rename('src_is_attacker')
ent     = pd.DataFrame({'out_degree':out_deg}).join(src_att, how='left')
ent['src_is_attacker'] = ent['src_is_attacker'].fillna(0).astype(int)
truth   = ent['src_is_attacker'].astype(bool)
od_nz   = ent.loc[ent['out_degree']>0,'out_degree'].astype(float)
s_thr   = float(od_nz.quantile(P/100))
S_out   = ent['out_degree'].astype(float) >= s_thr

print(f"\nEntities:{len(ent)} | Attackers:{int(truth.sum())} | "
      f"S_thr_P{P}:{s_thr:.1f} | S_flagged:{int(S_out.sum())}")

# ── 3) B channel ─────────────────────────────────────────
ben   = st[~st['is_attack']]
thr_b = {f: float(ben[f].quantile(P/100)) for f in FEATS_7}
bf    = np.zeros(len(st), dtype=bool)
for f,t in thr_b.items():
    bf |= st[f].to_numpy() >= t
B_col = (pd.DataFrame({'src_key':st['src_key'].to_numpy(),'B':bf.astype(int)})
           .groupby('src_key')['B'].max()
           .reindex(ent.index).fillna(0).astype(bool))

# ── 4) Isolation Forest ───────────────────────────────────
iso = IsolationForest(n_estimators=100, random_state=42, contamination='auto')
iso.fit(clean(st[FEATS_7]))
st['IF_flag'] = (iso.predict(clean(st[FEATS_7]))==-1).astype(int)
IF_col = (st.groupby('src_key')['IF_flag'].max()
            .reindex(ent.index).fillna(0).astype(bool))

# ── 5) Random Forest ──────────────────────────────────────
rf = RandomForestClassifier(n_estimators=50, random_state=42,
                             class_weight='balanced')
rf.fit(clean(sb[FEATS_7]), sb['is_attack'].astype(int))
st['RF_flag'] = rf.predict(clean(st[FEATS_7]))
RF_col = (st.groupby('src_key')['RF_flag'].max()
            .reindex(ent.index).fillna(0).astype(bool))

# ── 6) Sets ───────────────────────────────────────────────
Ss  = set(ent.index[S_out])
Bs  = set(B_col[B_col].index)
IFs = set(IF_col[IF_col].index)
RFs = set(RF_col[RF_col].index)
ATK = set(ent.index[truth])

# ── 7) Results ────────────────────────────────────────────
print('\n=== CROSS-PARADIGM INVARIANCE — NF-ToN-IoT-v2 ===')
print(f'S        = {Ss}')
print(f'S ∩ ATK  = {Ss & ATK}  |  ATK = {ATK}')
print(f'S ⊂ B         : {Ss.issubset(Bs)}')
print(f'S ⊂ IF        : {Ss.issubset(IFs)}')
print(f'S ⊂ B ∩ IF    : {Ss.issubset(Bs & IFs)}')
print(f'S ⊂ RF        : {Ss.issubset(RFs)}')
print(f'S ⊂ B ∩ RF ∩ IF: {Ss.issubset(Bs & RFs & IFs)}')

print('\n=== COMPARISON TABLE ===')
print(f"{'Environment':<25} {'GSD':>8} {'S⊂B':>6} {'S⊂IF':>6} {'S⊂B∩RF∩IF':>12}")
print('-'*60)
print(f"{'WTMC2021 [Ref-High]':<25} {'High':>8} {'Yes':>6} {'Yes':>6} {'Yes':>12}")
print(f"{'NF-ToN-IoT-v2':<25} {'Med/H':>8} "
      f"{'Yes' if Ss.issubset(Bs) else 'No':>6} "
      f"{'Yes' if Ss.issubset(IFs) else 'No':>6} "
      f"{'Yes' if Ss.issubset(Bs&RFs&IFs) else 'No':>12}")

# ── 8) Save ───────────────────────────────────────────────
with open(f'{OUTPUT_DIR}/cross_paradigm_ton_iot.json','w') as fout:
    json.dump({'environment':'NF-ToN-IoT-v2','GSD':'Medium-High',
               'S':list(Ss),'ATK':list(ATK),
               'S_subset_B':   Ss.issubset(Bs),
               'S_subset_IF':  Ss.issubset(IFs),
               'S_subset_B_IF':Ss.issubset(Bs & IFs),
               'S_subset_RF':  Ss.issubset(RFs),
               'S_subset_B_RF_IF': Ss.issubset(Bs&RFs&IFs)},
              fout, indent=2, default=str)
print('\n✓ Saved to Drive')

Loading 108 ToN IPs + 10 BoT IPs...
✓ ToN: 30,000 rows | 37 entities
✓ BoT: 30,000 rows | 10 entities

Entities:37 | Attackers:20 | S_thr_P95:162.2 | S_flagged:2

=== CROSS-PARADIGM INVARIANCE — NF-ToN-IoT-v2 ===
S        = {'NF-ToN-IoT-v2::192.168.1.32', 'NF-ToN-IoT-v2::192.168.1.31'}
S ∩ ATK  = {'NF-ToN-IoT-v2::192.168.1.32', 'NF-ToN-IoT-v2::192.168.1.31'}  |  ATK = {'NF-ToN-IoT-v2::192.168.1.30', 'NF-ToN-IoT-v2::192.168.1.33', 'NF-ToN-IoT-v2::192.168.1.28', 'NF-ToN-IoT-v2::192.168.1.194', 'NF-ToN-IoT-v2::192.168.1.152', 'NF-ToN-IoT-v2::192.168.1.31', 'NF-ToN-IoT-v2::192.168.1.38', 'NF-ToN-IoT-v2::192.168.1.193', 'NF-ToN-IoT-v2::192.168.1.79', 'NF-ToN-IoT-v2::192.168.1.133', 'NF-ToN-IoT-v2::192.168.1.184', 'NF-ToN-IoT-v2::192.168.1.35', 'NF-ToN-IoT-v2::192.168.1.190', 'NF-ToN-IoT-v2::192.168.1.39', 'NF-ToN-IoT-v2::192.168.1.32', 'NF-ToN-IoT-v2::192.168.1.36', 'NF-ToN-IoT-v2::192.168.1.37', 'NF-ToN-IoT-v2::192.168.1.169', 'NF-ToN-IoT-v2::192.168.1.250', 'NF-ToN-IoT-v2::192.168.1.34'}


## Step 10 — Robustness Check: 7-Feature Behavioral Channel
This section replicates the analysis with all 7 available features.
Results reported as supplementary/robustness evidence only.
Primary results are in Step 8 (temporal-only).

In [ ]:
BEHAVIOR_FEATURES_7 = [
    'FLOW_DURATION_MILLISECONDS','IN_BYTES','OUT_BYTES',
    'IN_PKTS','OUT_PKTS','SRC_TO_DST_SECOND_BYTES','DST_TO_SRC_SECOND_BYTES'
]

all_kdis_7 = []
for ds_name, sub in [('NF-UQ-NIDS-v2_COMBINED', df)] + [
        (str(n), s) for n, s in df.groupby(DATASET_COL) if len(s) >= 100]:
    res = run_kdis(sub.copy(), ds_name, BEHAVIOR_FEATURES_7)
    if not res.empty:
        all_kdis_7.append(res)

kdis_7 = pd.concat(all_kdis_7, ignore_index=True) if all_kdis_7 else pd.DataFrame()
hybrid_7 = kdis_7[kdis_7['Config']=='Hybrid'].copy()

print('=== ROBUSTNESS CHECK — 7 Features, Hybrid P95/P97 ===')
print(hybrid_7[
    hybrid_7['Threshold'].isin(['P95','P97'])
][['Dataset','Threshold','B_entity_rate','B_saturated',
   'Precision','Recall','FPR','Lift','RCI','RCI_Lift']]
 .round(4).to_string(index=False))

kdis_7.to_csv(f'{OUTPUT_DIR}/robustness_7features.csv',
              index=False, encoding='utf-8-sig')
print('\n✓ Robustness check saved')

KeyError: 'IN_BYTES'

## Step 11 — Final Summary & File Index

In [ ]:
files_saved = [
    ('gsd_summary_final.csv',       'GSD metrics — all environments + references'),
    ('kdis_full_results.csv',        'Full KDIS results — all configs & thresholds'),
    ('hybrid_results_final.csv',     'Hybrid table + corrected reference rows'),
    ('primary_results_p95_p97.csv',  'Primary results — non-saturated P95/P97 only'),
    ('robustness_7features.csv',     'Robustness check — 7-feature behavioral channel'),
    ('cross_paradigm_ton_iot.json',  'Cross-paradigm invariance — NF-ToN-IoT-v2'),
]

print(f'Output directory: {OUTPUT_DIR}')
print('\nFiles saved:')
for fname, desc in files_saved:
    path = f'{OUTPUT_DIR}/{fname}'
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1024 if exists else 0
    status = f'✓ ({size:.0f} KB)' if exists else '✗ missing'
    print(f'  {status:12} {fname:45} {desc}')

Output directory: /content/drive/MyDrive/paper2_gsd_results

Files saved:
  ✓ (1 KB)     gsd_summary_final.csv                         GSD metrics — all environments + references
  ✓ (7 KB)     kdis_full_results.csv                         Full KDIS results — all configs & thresholds
  ✓ (3 KB)     hybrid_results_final.csv                      Hybrid table + corrected reference rows
  ✓ (2 KB)     primary_results_p95_p97.csv                   Primary results — non-saturated P95/P97 only
  ✗ missing    robustness_7features.csv                      Robustness check — 7-feature behavioral channel
  ✓ (1 KB)     cross_paradigm_ton_iot.json                   Cross-paradigm invariance — NF-ToN-IoT-v2


In [ ]:
# Step 11 — File verification
import os

OUTPUT_DIR = '/content/drive/MyDrive/paper2_gsd_results'

files = [
    'gsd_summary_final.csv',
    'kdis_full_results.csv',
    'hybrid_results_final.csv',
    'primary_results_p95_p97.csv',
    'cross_paradigm_ton_iot.json',
]

print(f"Output: {OUTPUT_DIR}\n")
for f in files:
    path = f"{OUTPUT_DIR}/{f}"
    if os.path.exists(path):
        size = os.path.getsize(path)/1024
        print(f"  ✓  {f:<45} ({size:.0f} KB)")
    else:
        print(f"  ✗  {f} — missing")

Output: /content/drive/MyDrive/paper2_gsd_results

  ✓  gsd_summary_final.csv                         (1 KB)
  ✓  kdis_full_results.csv                         (7 KB)
  ✓  hybrid_results_final.csv                      (3 KB)
  ✓  primary_results_p95_p97.csv                   (2 KB)
  ✓  cross_paradigm_ton_iot.json                   (1 KB)
